# 🔧 Engenharia de Features para Previsão de Dengue

## Objetivo
Este notebook tem como objetivo criar features relevantes para a previsão de casos de dengue, incluindo:
- Features de lag (valores históricos)
- Médias móveis
- Features sazonais
- Agregações temporais
- Normalização e encoding de variáveis

## Estratégia
Trabalharemos a nível de estado (UF) para criar um modelo robusto que capture padrões regionais e temporais.

In [ ]:
# Importação das bibliotecas
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings
from sklearn.preprocessing import StandardScaler, LabelEncoder
import plotly.express as px
import plotly.graph_objects as go

warnings.filterwarnings('ignore')
plt.style.use('default')
sns.set_palette("husl")

print("✅ Bibliotecas carregadas com sucesso!")

In [ ]:
# Carregamento dos dados limpos
df = pd.read_csv('dados_dengue_limpos.csv')

# Recriação de colunas essenciais
df['Data'] = pd.to_datetime(df[['Ano', 'Mês']].assign(day=1))
uf_mapping = {
    'RO': 'Rondônia', 'AC': 'Acre', 'AM': 'Amazonas', 'RR': 'Roraima',
    'PA': 'Pará', 'AP': 'Amapá', 'TO': 'Tocantins', 'MA': 'Maranhão',
    'PI': 'Piauí', 'CE': 'Ceará', 'RN': 'Rio Grande do Norte', 'PB': 'Paraíba',
    'PE': 'Pernambuco', 'AL': 'Alagoas', 'SE': 'Sergipe', 'BA': 'Bahia',
    'MG': 'Minas Gerais', 'ES': 'Espírito Santo', 'RJ': 'Rio de Janeiro',
    'SP': 'São Paulo', 'PR': 'Paraná', 'SC': 'Santa Catarina',
    'RS': 'Rio Grande do Sul', 'MS': 'Mato Grosso do Sul', 'MT': 'Mato Grosso',
    'GO': 'Goiás', 'DF': 'Distrito Federal'
}
df['Estado'] = df['COD_UF'].map(uf_mapping)

# Ordenar por Estado e Data
df = df.sort_values(['COD_UF', 'Data']).reset_index(drop=True)

print(f"📋 Dataset carregado: {df.shape[0]} registros, {df.shape[1]} colunas")
print(f"📅 Período: {df['Data'].min().strftime('%Y-%m')} até {df['Data'].max().strftime('%Y-%m')}")
print(f"🗺️ Estados: {df['COD_UF'].nunique()}")

In [ ]:
# Função para criar features de lag
def create_lag_features(df, target_col, group_col, lags=[1, 2, 3, 6, 12]):
    """
    Cria features de lag para uma série temporal agrupada
    """
    df_lag = df.copy()

    for lag in lags:
        feature_name = f'{target_col}_lag_{lag}'
        df_lag[feature_name] = df_lag.groupby(group_col)[target_col].shift(lag)
        print(f"✅ Feature criada: {feature_name}")

    return df_lag

# Criar features de lag para casos de dengue
print("🔄 Criando features de lag...")
df = create_lag_features(df, 'Quantidade de Casos', 'COD_UF', lags=[1, 2, 3, 6, 12])

print(f"\n📊 Novas dimensões: {df.shape}")

In [ ]:
# Função para criar médias móveis
def create_rolling_features(df, target_col, group_col, windows=[3, 6, 12]):
    """
    Cria features de médias móveis para uma série temporal agrupada
    """
    df_roll = df.copy()

    for window in windows:
        # Média móvel
        feature_name = f'{target_col}_rolling_mean_{window}'
        df_roll[feature_name] = df_roll.groupby(group_col)[target_col].transform(
            lambda x: x.rolling(window=window, min_periods=1).mean()
        )

        # Desvio padrão móvel
        feature_name_std = f'{target_col}_rolling_std_{window}'
        df_roll[feature_name_std] = df_roll.groupby(group_col)[target_col].transform(
            lambda x: x.rolling(window=window, min_periods=1).std()
        )

        print(f"✅ Features criadas: {feature_name} e {feature_name_std}")

    return df_roll

# Criar features de médias móveis
print("📈 Criando features de médias móveis...")
df = create_rolling_features(df, 'Quantidade de Casos', 'COD_UF', windows=[3, 6, 12])

print(f"\n📊 Novas dimensões: {df.shape}")

In [ ]:
# Features sazonais e temporais
print("📅 Criando features temporais e sazonais...")

# Features básicas de tempo
df['Trimestre'] = df['Data'].dt.quarter
df['Dia_Ano'] = df['Data'].dt.dayofyear
df['Semestre'] = np.where(df['Mês'] <= 6, 1, 2)

# Features cíclicas (usando seno e cosseno para capturar a natureza cíclica)
df['Mês_Sin'] = np.sin(2 * np.pi * df['Mês'] / 12)
df['Mês_Cos'] = np.cos(2 * np.pi * df['Mês'] / 12)
df['Trimestre_Sin'] = np.sin(2 * np.pi * df['Trimestre'] / 4)
df['Trimestre_Cos'] = np.cos(2 * np.pi * df['Trimestre'] / 4)

# Estações do ano (baseado no hemisfério sul)
def get_season(month):
    if month in [12, 1, 2]:
        return 'Verão'
    elif month in [3, 4, 5]:
        return 'Outono'
    elif month in [6, 7, 8]:
        return 'Inverno'
    else:
        return 'Primavera'

df['Estacao'] = df['Mês'].apply(get_season)

# Período de chuvas (varia por região, mas generalizando)
df['Periodo_Chuvas'] = np.where(df['Mês'].isin([10, 11, 12, 1, 2, 3]), 1, 0)

print(f"✅ Features temporais criadas!")
print(f"📊 Novas dimensões: {df.shape}")

In [ ]:
# Features de interação climática
print("🌡️ Criando features de interação climática...")

# Amplitude térmica
df['Amplitude_Termica'] = df['temp_max_media_mensal_uf'] - df['temp_min_media_mensal_uf']

# Amplitude de umidade
df['Amplitude_Umidade'] = df['umidade_max_media_mensal_uf'] - df['umidade_min_media_mensal_uf']

# Amplitude de pressão
df['Amplitude_Pressao'] = df['pressao_max_media_mensal_uf'] - df['pressao_min_media_mensal_uf']

# Índice de conforto térmico
df['Temp_Media'] = (df['temp_max_media_mensal_uf'] + df['temp_min_media_mensal_uf']) / 2
df['Umidade_Media'] = (df['umidade_max_media_mensal_uf'] + df['umidade_min_media_mensal_uf']) / 2

# Interações importantes para dengue
df['Precipitacao_Temp'] = df['precipitacao_media_mensal_uf'] * df['Temp_Media']
df['Precipitacao_Umidade'] = df['precipitacao_media_mensal_uf'] * df['Umidade_Media']
df['Temp_Umidade'] = df['Temp_Media'] * df['Umidade_Media']

print(f"✅ Features de interação criadas!")
print(f"📊 Novas dimensões: {df.shape}")

In [ ]:
# Features demográficas e de saneamento
print("🏙️ Criando features demográficas...")

# Taxa de urbanização
df['Taxa_Urbanizacao'] = df['População urbana(pessoas)'] / df['População(pessoas)']

# Pessoas por km² urbano (aproximação)
df['Densidade_Urbana_Estimada'] = df['População urbana(pessoas)'] / (df['Densidade demográfica(Pessoas por km²)'] + 1)

# Log das variáveis populacionais (para reduzir assimetria)
df['Log_Populacao'] = np.log1p(df['População(pessoas)'])
df['Log_Populacao_Urbana'] = np.log1p(df['População urbana(pessoas)'])

# Gasto per capita em saneamento normalizado
df['Saneamento_Normalizado'] = df['Despesas per capita com saneamento, R$ a preços de 2024(R$ per capita a preços de 2024)'] / df['Taxa_Urbanizacao']

print(f"✅ Features demográficas criadas!")
print(f"📊 Novas dimensões: {df.shape}")

In [ ]:
# Features de tendência temporal
print("📊 Criando features de tendência...")

# Tendência linear por estado
df['Mes_Sequencial'] = df.groupby('COD_UF').cumcount() + 1

# Taxa de crescimento dos casos (mês a mês)
df['Casos_Taxa_Crescimento'] = df.groupby('COD_UF')['Quantidade de Casos'].pct_change()

# Diferença dos casos em relação ao mês anterior
df['Casos_Diff'] = df.groupby('COD_UF')['Quantidade de Casos'].diff()

# Média histórica do mesmo mês em anos anteriores
df['Casos_Media_Historica_Mes'] = df.groupby(['COD_UF', 'Mês'])['Quantidade de Casos'].transform(
    lambda x: x.expanding().mean().shift(1)
)

print(f"✅ Features de tendência criadas!")
print(f"📊 Novas dimensões: {df.shape}")

In [ ]:
# Encoding de variáveis categóricas
print("🔤 Realizando encoding de variáveis categóricas...")

# Label encoding para variáveis ordinais
le_estacao = LabelEncoder()
df['Estacao_Encoded'] = le_estacao.fit_transform(df['Estacao'])

# One-hot encoding para estados (ou usar target encoding se necessário)
estado_dummies = pd.get_dummies(df['COD_UF'], prefix='Estado')
df = pd.concat([df, estado_dummies], axis=1)

print(f"✅ Encoding realizado!")
print(f"📊 Dimensões finais: {df.shape}")

In [ ]:
# Análise das novas features
print("🔍 Analisando correlações das novas features...")

# Listar todas as features numéricas criadas
features_numericas = df.select_dtypes(include=[np.number]).columns
features_criadas = [col for col in features_numericas if
                   any(keyword in col for keyword in ['lag', 'rolling', 'Sin', 'Cos', 'Amplitude',
                                                     'Media', 'Taxa', 'Log', 'Diff', 'Sequencial'])]

print(f"\n📈 Features criadas ({len(features_criadas)}):")
for feature in features_criadas:
    print(f"   • {feature}")

# Correlação das novas features com o target
correlations = df[features_criadas + ['Quantidade de Casos']].corr()['Quantidade de Casos'].abs().sort_values(ascending=False)

print(f"\n🎯 Top 10 features mais correlacionadas com casos de dengue:")
for i, (feature, corr) in enumerate(correlations.drop('Quantidade de Casos').head(10).items(), 1):
    print(f"   {i:2d}. {feature}: {corr:.3f}")

In [ ]:
# Visualização das features mais importantes
top_features = correlations.drop('Quantidade de Casos').head(8).index

fig, axes = plt.subplots(2, 4, figsize=(20, 10))
axes = axes.ravel()

for i, feature in enumerate(top_features):
    # Remove valores infinitos e NaN para a visualização
    mask = np.isfinite(df[feature]) & np.isfinite(df['Quantidade de Casos'])
    x_data = df.loc[mask, feature]
    y_data = df.loc[mask, 'Quantidade de Casos']

    axes[i].scatter(x_data, y_data, alpha=0.5, s=20)
    axes[i].set_xlabel(feature.replace('_', ' '))
    axes[i].set_ylabel('Casos de Dengue')
    axes[i].set_title(f'{feature}\nCorr: {correlations[feature]:.3f}')

plt.tight_layout()
plt.show()

In [ ]:
# Tratamento de valores ausentes e infinitos
print("🧹 Tratando valores ausentes e infinitos...")

# Identificar colunas com valores ausentes
missing_counts = df.isnull().sum()
cols_with_missing = missing_counts[missing_counts > 0]

if len(cols_with_missing) > 0:
    print(f"Colunas com valores ausentes:")
    for col, count in cols_with_missing.items():
        print(f"   • {col}: {count} ({count/len(df)*100:.1f}%)")

# Tratamento de valores infinitos
for col in df.select_dtypes(include=[np.number]).columns:
    if np.isinf(df[col]).any():
        print(f"Valores infinitos encontrados em: {col}")
        df[col] = df[col].replace([np.inf, -np.inf], np.nan)

# Preenchimento de valores ausentes com forward fill e backward fill por grupo
numeric_cols = df.select_dtypes(include=[np.number]).columns
for col in numeric_cols:
    if df[col].isnull().any():
        df[col] = df.groupby('COD_UF')[col].transform(lambda x: x.fillna(method='ffill').fillna(method='bfill'))
        # Se ainda houver NaN, usar a mediana global
        if df[col].isnull().any():
            df[col] = df[col].fillna(df[col].median())

print(f"✅ Tratamento concluído!")
print(f"📊 Valores ausentes restantes: {df.isnull().sum().sum()}")

In [ ]:
# Identificar e preparar features para modelagem
print("🎯 Preparando features para modelagem...")

# Features a serem excluídas da modelagem
exclude_cols = [
    'Quantidade de Casos',  # Target
    'Data',  # Data (usaremos features derivadas)
    'Estado',  # String (usaremos dummy variables)
    'Estacao',  # String (usaremos encoding)
]

# Selecionar features para modelagem
feature_cols = [col for col in df.columns if col not in exclude_cols]

# Remover colunas com variância zero
variance_filter = df[feature_cols].var() > 0
feature_cols = [col for col in feature_cols if variance_filter[col]]

print(f"📊 Total de features para modelagem: {len(feature_cols)}")

# Criar DataFrame final
X_features = df[feature_cols]
y_target = df['Quantidade de Casos']

print(f"✅ Dataset de features: {X_features.shape}")
print(f"✅ Target: {y_target.shape}")

In [ ]:
# Análise final das features
print("📋 RESUMO DAS FEATURES CRIADAS:")
print("="*50)

feature_categories = {
    'Lag Features': [col for col in feature_cols if 'lag' in col],
    'Rolling Features': [col for col in feature_cols if 'rolling' in col],
    'Temporal Features': [col for col in feature_cols if any(x in col for x in ['Sin', 'Cos', 'Trimestre', 'Mes'])],
    'Climate Features': [col for col in feature_cols if any(x in col for x in ['temp', 'precipitacao', 'umidade', 'pressao', 'Amplitude'])],
    'Demographic Features': [col for col in feature_cols if any(x in col for x in ['População', 'Densidade', 'Taxa', 'Log'])],
    'State Features': [col for col in feature_cols if 'Estado_' in col],
    'Other Features': [col for col in feature_cols if not any(cat_name in str(col) for cat_name in ['lag', 'rolling', 'Sin', 'Cos', 'Trimestre', 'Mes', 'temp', 'precipitacao', 'umidade', 'pressao', 'Amplitude', 'População', 'Densidade', 'Taxa', 'Log', 'Estado_'])]
}

for category, features in feature_categories.items():
    if features:
        print(f"\n{category} ({len(features)}):")
        for feature in features[:5]:  # Mostrar apenas os primeiros 5
            print(f"   • {feature}")
        if len(features) > 5:
            print(f"   • ... e mais {len(features)-5}")

print(f"\n🎯 ESTATÍSTICAS FINAIS:")
print(f"   • Total de registros: {len(df):,}")
print(f"   • Total de features: {len(feature_cols)}")
print(f"   • Período: {df['Data'].min().strftime('%Y-%m')} até {df['Data'].max().strftime('%Y-%m')}")
print(f"   • Estados: {df['COD_UF'].nunique()}")
print(f"   • Valores ausentes: {df[feature_cols].isnull().sum().sum()}")

In [ ]:
# Salvar datasets processados
print("💾 Salvando datasets processados...")

# Dataset completo com features
df.to_csv('dados_com_features.csv', index=False)
print("✅ Dataset completo salvo: 'dados_com_features.csv'")

# Features e target separados
X_features.to_csv('X_features.csv', index=False)
y_target.to_csv('y_target.csv', index=False, header=['Quantidade_Casos'])
print("✅ Features salvas: 'X_features.csv'")
print("✅ Target salvo: 'y_target.csv'")

# Lista de features para referência
with open('lista_features.txt', 'w', encoding='utf-8') as f:
    f.write("LISTA DE FEATURES PARA MODELAGEM\n")
    f.write("="*40 + "\n\n")

    for category, features in feature_categories.items():
        if features:
            f.write(f"{category} ({len(features)}):\n")
            for feature in features:
                f.write(f"   • {feature}\n")
            f.write("\n")

print("✅ Lista de features salva: 'lista_features.txt'")

print("\n🎉 ENGENHARIA DE FEATURES CONCLUÍDA!")
print("Próximo passo: Modelagem e treinamento dos algoritmos")